In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path
from urllib.request import urlopen, urlretrieve

from IPython.display import Markdown, display


# ============================================================
# 0. Branch / repo config
# ============================================================

BRANCH = "Atlas"
REPO = "onestardao/WFGY"
BASE_RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/ProblemMap/Atlas/Fixes/official/demos"
SHARED_RAW = f"{BASE_RAW}/shared"
DEMO_RAW = f"{BASE_RAW}/demo-f7-container-fidelity"

WORKDIR = Path("/content/wfgy_demo4_shared_runtime")
SHARED_DIR = WORKDIR / "shared"
WORKDIR.mkdir(parents=True, exist_ok=True)
SHARED_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. Shared-layer import
# ============================================================

def import_module_from_file(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


SHARED_OK = False
display_helpers = None
demo_utils = None
shared_status_note = "Shared helper layer was not loaded. Fallback display mode is active."

try:
    for filename in ("demo_utils.py", "display_helpers.py"):
        urlretrieve(f"{SHARED_RAW}/{filename}", SHARED_DIR / filename)

    demo_utils = import_module_from_file("wfgy_demo_utils", SHARED_DIR / "demo_utils.py")
    display_helpers = import_module_from_file("wfgy_display_helpers", SHARED_DIR / "display_helpers.py")
    SHARED_OK = True
    shared_status_note = "Shared helper layer loaded successfully from the Atlas branch."
except Exception as exc:
    shared_status_note = f"Shared helper layer not loaded. Fallback display mode is active. Reason: {type(exc).__name__}"


# ============================================================
# 2. Fallback display helpers
# ============================================================

def md(text: str):
    display(Markdown(text))


def fallback_section_title(title: str, level: int = 2):
    level = max(1, min(level, 6))
    md(f"{'#' * level} {title}")


def fallback_callout(text: str):
    md(f"> {text}")


def fallback_mode_note(mode_label: str, note: str | None = None):
    lines = [
        "### Mode",
        "",
        f"- **Current mode:** {mode_label}",
    ]
    if note:
        lines.append(f"- **Note:** {note}")
    md("\n".join(lines))


def fallback_json_card(title: str, data):
    md(f"### {title}\n\n```json\n{json.dumps(data, indent=2, ensure_ascii=False)}\n```")


def fallback_route_card(primary_family: str, secondary_family: str, best_current_fit: str, broken_invariant: str, title: str = "Route Summary"):
    md(
        "\n".join(
            [
                f"### {title}",
                "",
                f"- **Primary family:** {primary_family}",
                f"- **Secondary family:** {secondary_family}",
                f"- **Best current fit:** {best_current_fit}",
                f"- **Broken invariant:** {broken_invariant}",
            ]
        )
    )


def fallback_before_after_card(before, after, before_label: str = "Before", after_label: str = "After", title: str = "Before / After"):
    if isinstance(before, (dict, list)):
        before_text = json.dumps(before, indent=2, ensure_ascii=False)
    else:
        before_text = str(before)

    if isinstance(after, (dict, list)):
        after_text = json.dumps(after, indent=2, ensure_ascii=False)
    else:
        after_text = str(after)

    md(
        "\n".join(
            [
                f"### {title}",
                "",
                f"#### {before_label}",
                "",
                "```text",
                before_text,
                "```",
                "",
                f"#### {after_label}",
                "",
                "```text",
                after_text,
                "```",
            ]
        )
    )


def fallback_checklist(title: str, items):
    lines = [f"### {title}", ""]
    for item in items:
        lines.append(f"- [x] {item}")
    md("\n".join(lines))


if not SHARED_OK:
    class FallbackDisplayHelpers:
        display_section_title = staticmethod(fallback_section_title)
        display_callout = staticmethod(fallback_callout)
        display_mode_note = staticmethod(fallback_mode_note)
        display_json_card = staticmethod(fallback_json_card)
        display_route_card = staticmethod(fallback_route_card)
        display_before_after_card = staticmethod(fallback_before_after_card)
        display_checklist = staticmethod(fallback_checklist)

    display_helpers = FallbackDisplayHelpers()


# ============================================================
# 3. Generic loaders
# ============================================================

def load_json_from_url(url: str):
    with urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))


if SHARED_OK and hasattr(demo_utils, "load_json"):
    input_case = demo_utils.load_json(f"{DEMO_RAW}/input_case.json")
    replay_outputs = demo_utils.load_json(f"{DEMO_RAW}/replay_outputs.json")
    expected_output = demo_utils.load_json(f"{DEMO_RAW}/expected_output.json")
else:
    input_case = load_json_from_url(f"{DEMO_RAW}/input_case.json")
    replay_outputs = load_json_from_url(f"{DEMO_RAW}/replay_outputs.json")
    expected_output = load_json_from_url(f"{DEMO_RAW}/expected_output.json")


# ============================================================
# 4. Small extraction helpers
# ============================================================

def pick_existing_keys(data: dict, keys: list[str]) -> dict:
    out = {}
    for key in keys:
        if key in data and data[key] not in (None, "", [], {}):
            out[key] = data[key]
    return out


def first_non_empty(*values):
    for value in values:
        if value not in (None, "", [], {}):
            return value
    return None


family_target = input_case.get("family_target", {})
minimum_success_contract = expected_output.get("minimum_success_contract", expected_output)

primary_family = first_non_empty(
    family_target.get("primary_family"),
    expected_output.get("primary_family"),
    minimum_success_contract.get("primary_family"),
    "F7",
)

secondary_family = first_non_empty(
    family_target.get("secondary_family"),
    expected_output.get("secondary_family"),
    minimum_success_contract.get("secondary_family"),
    "F2",
)

best_current_fit = first_non_empty(
    family_target.get("best_current_fit"),
    expected_output.get("best_current_fit"),
    minimum_success_contract.get("best_current_fit"),
    "F7_N01_B Formal Container Adequacy Failure",
)

broken_invariant = first_non_empty(
    family_target.get("broken_invariant"),
    expected_output.get("broken_invariant"),
    minimum_success_contract.get("broken_invariant"),
    "representation_container_fidelity_broken",
)

summary_block = first_non_empty(
    replay_outputs.get("summary"),
    replay_outputs.get("replay_summary"),
    {},
)

route_replay = first_non_empty(
    replay_outputs.get("route_replay"),
    replay_outputs.get("routing_explanation"),
    {},
)

before_after = first_non_empty(
    replay_outputs.get("before_after_comparison"),
    replay_outputs.get("before_after"),
    {},
)

before_state = first_non_empty(
    before_after.get("before") if isinstance(before_after, dict) else None,
    replay_outputs.get("before"),
    replay_outputs.get("baseline_state"),
    {},
)

after_state = first_non_empty(
    before_after.get("after") if isinstance(before_after, dict) else None,
    replay_outputs.get("after"),
    replay_outputs.get("repaired_state"),
    {},
)

what_changed = first_non_empty(
    before_after.get("what_changed") if isinstance(before_after, dict) else None,
    replay_outputs.get("what_changed"),
    [],
)

new_structural_value = first_non_empty(
    replay_outputs.get("improved_structure_snapshot", {}).get("new_structural_value") if isinstance(replay_outputs.get("improved_structure_snapshot"), dict) else None,
    replay_outputs.get("improved_state_signals"),
    replay_outputs.get("new_structural_value"),
    [],
)

baseline_diag = first_non_empty(
    replay_outputs.get("baseline_diagnosis"),
    replay_outputs.get("baseline_analysis"),
    replay_outputs.get("baseline_output"),
    {},
)

repaired_diag = first_non_empty(
    replay_outputs.get("repaired_diagnosis"),
    replay_outputs.get("repaired_analysis"),
    replay_outputs.get("repaired_output"),
    {},
)

case_overview = pick_existing_keys(
    input_case,
    [
        "demo_id",
        "demo_version",
        "case_id",
        "title",
        "task_type",
        "user_question",
        "structure_target",
    ],
)

structure_target = pick_existing_keys(
    input_case,
    [
        "intended_structure",
        "structure_target",
        "target_schema",
        "baseline_container",
        "baseline_shell",
        "formal_target",
        "descriptor_context",
        "representation_context",
        "symbolic_constraints",
        "layout_constraints",
        "schema_requirements",
    ],
)

if not structure_target:
    structure_target = {
        key: value
        for key, value in input_case.items()
        if key not in {"demo_id", "demo_version", "case_id", "title", "task_type", "user_question", "family_target"}
    }


# ============================================================
# 5. Render the notebook
# ============================================================

display_helpers.display_section_title("Demo 4 · F7 Container Fidelity", level=1)
display_helpers.display_callout(
    "Replay-only MVP notebook. This version keeps the official replay teaching flow compact while adding a clearer route summary, stronger structure-first framing, and a cleaner shared-layer display path."
)
display_helpers.display_mode_note(
    "Replay-only MVP",
    "No API key required. The first goal is to show that some reasoning-looking failures should be repaired through container fidelity first, not through generic reasoning pressure."
)

display_helpers.display_json_card("Case Overview", case_overview)

display_helpers.display_route_card(
    primary_family=primary_family,
    secondary_family=secondary_family,
    best_current_fit=best_current_fit,
    broken_invariant=broken_invariant,
    title="Atlas Route Summary",
)

display_helpers.display_json_card("Structure Target", structure_target)

if summary_block:
    display_helpers.display_json_card("Replay Summary", summary_block)

if route_replay:
    display_helpers.display_json_card("Why F7 Is Primary", route_replay)

if baseline_diag not in (None, "", [], {}):
    display_helpers.display_json_card(
        "Baseline Replay Diagnosis",
        baseline_diag if isinstance(baseline_diag, (dict, list)) else {"baseline_diagnosis": baseline_diag},
    )

if repaired_diag not in (None, "", [], {}):
    display_helpers.display_json_card(
        "Repaired Replay Diagnosis",
        repaired_diag if isinstance(repaired_diag, (dict, list)) else {"repaired_diagnosis": repaired_diag},
    )

display_helpers.display_before_after_card(
    before=before_state,
    after=after_state,
    before_label="Before",
    after_label="After",
    title="Replay Before / After",
)

if what_changed or new_structural_value:
    display_helpers.display_json_card(
        "What Changed",
        {
            "what_changed": what_changed,
            "new_structural_value": new_structural_value,
        },
    )

display_helpers.display_json_card(
    "Expected Minimum Success Contract",
    minimum_success_contract,
)

display_helpers.display_checklist(
    "Quick Validation Checklist",
    [
        "The notebook clearly presents Demo 4 as replay-only MVP",
        "The atlas route summary is visible",
        "The structure target is visible",
        "The baseline and repaired replay diagnoses are visible",
        "The before / after structural shift is easy to inspect",
        "The minimum success contract is visible",
        "The notebook is readable without any API key",
    ],
)

display_helpers.display_json_card(
    "Shared Layer Status",
    {
        "shared_layer_loaded": SHARED_OK,
        "note": shared_status_note,
        "working_directory": str(WORKDIR),
    },
)

display_helpers.display_callout(
    "Main lesson: the first useful repair is container repair, not premature reasoning pressure. If the structure carrier is weak, repair the carrier first."
)

md(
    """
## Back to the main page

Read the full product page here:
[Problem Map 3.0 Troubleshooting Atlas](https://github.com/onestardao/WFGY/blob/main/ProblemMap/wfgy-ai-problem-map-troubleshooting-atlas.md)

If you like the project, star the repo ⭐
"""
)